# Bell-Plate FEM — interactive modal analysis

Free-vibration (modal) analysis of a rectangular **bell plate** (a struck metal idiophone)
with optional **triangular edge incuts**, using 3D solid finite elements
(gmsh mesh → scikit-fem P2 tetrahedra → free-free eigenvalue solve → matplotlib mode shapes).

**How to use:** run **all cells once** (menu ▸ *Run All*), then use the control panel at the
bottom — set the plate dimensions, material, and incuts, and press **▶ Run** to compute and
see the results. No need to edit any code.

### Install (run once, if imports fail)
```
pip install numpy scipy scikit-fem meshio gmsh matplotlib ipywidgets
```

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.sparse.linalg import eigsh

import os, tempfile, traceback
import gmsh
import skfem
from skfem import MeshTet, Basis, ElementVector, ElementTetP2, BilinearForm
from skfem.helpers import ddot, sym_grad, trace, dot
import ipywidgets as W
from IPython.display import display

print("scikit-fem", skfem.__version__, "| gmsh", gmsh.__version__, "| ipywidgets", W.__version__)

## Model details

- **Boundary condition:** *free-free* (unconstrained, as if suspended at nodal points and
  struck). A free body has **6 rigid-body modes** at ~0 Hz, detected and filtered out.
- **Units:** everything is SI internally (m, Pa, kg/m³); you enter mm / GPa, converted on the way in.
- **Mesh sizing** auto-derives the element size from `f_max` (flexural wavelength ÷ density),
  capped to **one quadratic (P2) element through the thickness** (within ~1% of a finer mesh but
  several times faster) and a couple of elements across each notch. Use the convergence check to confirm.
- **Incuts:** isosceles V-notches cut through the full thickness, boolean-subtracted in gmsh.

In [ ]:
# Material presets: E [Pa], rho [kg/m^3], nu [-]
MATERIALS = {
    "steel":    dict(E=2.10e11, rho=7850.0, nu=0.30),
    "bronze":   dict(E=1.05e11, rho=8800.0, nu=0.34),
    "aluminum": dict(E=6.90e10, rho=2700.0, nu=0.33),
}

def flexural_wavelength(f, E, rho, nu, H):
    '''Bending wavelength of a plate of thickness H at frequency f (thin-plate estimate).'''
    D = E*H**3 / (12.0*(1.0 - nu**2))
    omega = 2.0*np.pi*f
    kappa = (omega**2 * rho*H / D)**0.25       # D kappa^4 = rho H omega^2
    return 2.0*np.pi/kappa

def compute_element_size(L, W_, H, E, rho, nu, f_max, epw, notches_m):
    '''Choose the mesh element size; return (h_elem, governing_rule, printable_lines).

    Sizing rules (smallest wins):
      - in-plane: flexural wavelength at f_max / epw (resolves the bending waves);
      - thickness: <= H, i.e. one QUADRATIC (P2) element through the thickness. A P2 element
        has a mid-edge node, so a single element already captures the through-thickness bending
        of the low modes -- verified within ~1% of a 2-elements-through-thickness mesh while
        solving several times faster. (Use the convergence check for a finer confirmation.)
      - notches: <= min(base, depth) / 2, so each notch spans >= ~2 elements.'''
    lam     = flexural_wavelength(f_max, E, rho, nu, H)
    h_wave  = lam / epw
    h_thick = H                                  # one P2 element through the thickness
    h = min(h_wave, h_thick)
    rule = "thickness cap (H)" if h_thick <= h_wave else f"{epw}/flexural wavelength"
    lines = [f"Flexural wavelength @ {f_max:g} Hz : {lam*1e3:7.1f} mm",
             f"  wavelength rule  -> h_wave    : {h_wave*1e3:7.2f} mm",
             f"  thickness rule   -> H         : {h_thick*1e3:7.2f} mm"]
    if notches_m:
        feat = min(min(n['base'], n['depth']) for n in notches_m)
        h_feat = feat / 2.0
        lines.append(f"  notch rule       -> feat/2    : {h_feat*1e3:7.2f} mm")
        if h_feat < h:
            h, rule = h_feat, "notch feature (min(base,depth)/2)"
    n_est = (L/h)*(W_/h)*(H/h)*6.0
    lines.append(f"  chosen element size  h_elem   : {h*1e3:7.2f} mm   (governed by {rule})")
    lines.append(f"  rough element count           : ~{n_est:,.0f} tets")
    return h, rule, lines

In [ ]:
def notch_prism(pos, base, depth, edge, L, W_, H):
    '''Build one triangular-prism cutting tool in the active gmsh model; return its (3, tag).'''
    if   edge == "bottom": O = np.array([pos, 0.0]); n = np.array([0.0,  1.0]); t = np.array([1.0, 0.0])
    elif edge == "top":    O = np.array([pos, W_]);  n = np.array([0.0, -1.0]); t = np.array([1.0, 0.0])
    elif edge == "left":   O = np.array([0.0, pos]); n = np.array([ 1.0, 0.0]); t = np.array([0.0, 1.0])
    elif edge == "right":  O = np.array([L, pos]);   n = np.array([-1.0, 0.0]); t = np.array([0.0, 1.0])
    else: raise ValueError(f"unknown edge {edge!r}")

    over = 0.05*depth                          # overhang so the base pokes outside the plate
    A  = O + depth*n                           # apex (inside the plate)
    P1 = O - 0.5*base*t                        # base endpoints at the surface
    P2 = O + 0.5*base*t
    s  = (depth + over)/depth                  # scale base outward from the apex
    P1o, P2o = A + s*(P1 - A), A + s*(P2 - A)

    pt = [gmsh.model.occ.addPoint(p[0], p[1], -0.05*H) for p in (A, P1o, P2o)]
    ln = [gmsh.model.occ.addLine(pt[i], pt[(i+1) % 3]) for i in range(3)]
    surf = gmsh.model.occ.addPlaneSurface([gmsh.model.occ.addCurveLoop(ln)])
    out = gmsh.model.occ.extrude([(2, surf)], 0.0, 0.0, H + 0.10*H)   # through the top face
    return [e for e in out if e[0] == 3][0]

def make_mesh(L, W_, H, h, notches=None):
    '''Mesh an L x W x H box (optionally with triangular edge notches cut out); return MeshTet.

    Raises a clear RuntimeError if the notch geometry leaves no valid solid, or if meshing
    produces no tetrahedra, instead of failing obscurely downstream.'''
    gmsh.initialize()
    try:
        gmsh.option.setNumber("General.Terminal", 0)
        gmsh.model.add("plate")
        box = gmsh.model.occ.addBox(0.0, 0.0, 0.0, L, W_, H)
        if notches:
            tools = [notch_prism(nc["pos"], nc["base"], nc["depth"], nc["edge"], L, W_, H)
                     for nc in notches]
            gmsh.model.occ.cut([(3, box)], tools, removeTool=True)
        gmsh.model.occ.synchronize()
        if len(gmsh.model.getEntities(3)) == 0:
            raise RuntimeError("The notches removed the entire plate (no solid left) — "
                               "reduce notch depth/base, or move overlapping notches apart.")
        gmsh.option.setNumber("Mesh.MeshSizeMin", h)
        gmsh.option.setNumber("Mesh.MeshSizeMax", h)
        gmsh.model.mesh.generate(3)
        path = os.path.join(tempfile.gettempdir(), "bell_plate.msh")
        gmsh.write(path)
    finally:
        gmsh.finalize()
    mesh = MeshTet.load(path)
    if mesh.t.shape[1] == 0:
        raise RuntimeError("Meshing produced no tetrahedra — check the plate/notch dimensions.")
    return mesh

In [ ]:
def solve_modes(mesh, E, rho, nu, n_modes, f_max):
    '''Assemble 3D elasticity (P2) and solve the free-free modal problem.

    Returns (basis, freqs, vecs) sorted ascending, INCLUDING the 6 rigid-body modes.'''
    mu  = E / (2.0*(1.0 + nu))
    lam = E*nu / ((1.0 + nu)*(1.0 - 2.0*nu))
    basis = Basis(mesh, ElementVector(ElementTetP2()))

    @BilinearForm
    def stiffness(u, v, w):
        return 2.0*mu*ddot(sym_grad(u), sym_grad(v)) + lam*trace(sym_grad(u))*trace(sym_grad(v))

    @BilinearForm
    def mass(u, v, w):
        return rho*dot(u, v)

    K = stiffness.assemble(basis)
    M = mass.assemble(basis)

    sigma = -(2.0*np.pi*max(1.0, 0.05*f_max))**2   # negative -> (K - sigma M) is SPD
    k = min(n_modes + 6 + 4, K.shape[0] - 2)
    if k < 1:
        raise RuntimeError("Mesh is too small to solve for any modes — use a finer mesh.")

    # eigsh (ARPACK shift-invert) occasionally fails to converge; retry once with a larger
    # Krylov subspace and more iterations before giving up with a clear message.
    try:
        vals, vecs = eigsh(K, k=k, M=M, sigma=sigma, which="LM")
    except Exception:
        ncv = min(K.shape[0] - 1, max(2*k + 1, 40))
        try:
            vals, vecs = eigsh(K, k=k, M=M, sigma=sigma, which="LM", ncv=ncv, maxiter=5000)
        except Exception as err:
            raise RuntimeError(
                f"Eigensolver did not converge (k={k}, ndof={K.shape[0]}). "
                f"Try fewer modes, or a coarser/finer mesh. [{type(err).__name__}: {err}]")

    order = np.argsort(vals)
    vals, vecs = vals[order], vecs[:, order]
    freqs = np.sqrt(np.clip(vals, 0.0, None)) / (2.0*np.pi)
    return basis, freqs, vecs

def split_rigid_elastic(freqs, n_modes):
    '''Separate the 6 rigid-body modes (f ~ 0) from elastic modes by a relative threshold.

    Robust to receiving fewer than 7 modes; returns (n_rigid, elastic_index_array).'''
    freqs = np.asarray(freqs, dtype=float)
    if freqs.size == 0:
        return 0, np.array([], dtype=int)
    ref = freqs[6] if freqs.size > 6 else freqs[-1]          # first clearly-elastic frequency
    thr = max(1e-3, 0.01 * ref)                              # rigid modes sit far below this
    rigid_mask = freqs < thr
    elastic_idx = np.where(~rigid_mask)[0][:n_modes]
    return int(rigid_mask.sum()), elastic_idx

def elastic_frequencies(mesh, E, rho, nu, n_modes, f_max):
    '''Convenience: solve and return just the elastic frequency array.'''
    _, freqs, _ = solve_modes(mesh, E, rho, nu, n_modes, f_max)
    _, idx = split_rigid_elastic(freqs, n_modes)
    return freqs[idx]

In [ ]:
def _top_surface_triangulation(mesh, H):
    '''Triangulation of the top (z=H) surface from real mesh facets (respects notch cut-outs).'''
    f = mesh.facets
    top = np.all(np.abs(mesh.p[2][f] - H) < 1e-6*H, axis=0)
    facets = f[:, top]
    nodes = np.unique(facets)
    remap = -np.ones(mesh.p.shape[1], dtype=int)
    remap[nodes] = np.arange(nodes.size)
    tri = mtri.Triangulation(mesh.p[0][nodes], mesh.p[1][nodes], remap[facets].T)
    return nodes, tri

def _draw_contour(ax, mesh, uzn, H):
    '''2D filled contour of normalised u_z on the top surface (nodal lines in black).'''
    nodes, tri = _top_surface_triangulation(mesh, H)
    norm = plt.Normalize(-1.0, 1.0)
    levels = np.linspace(-1.0, 1.0, 21)          # symmetric -> white exactly at 0
    ax.tricontourf(tri, uzn[nodes], levels=levels, cmap="RdBu_r", norm=norm, extend="both")
    ax.tricontour(tri, uzn[nodes], levels=[0.0], colors="k", linewidths=0.9)
    ax.set_aspect("equal")
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    ax.set_title("top-surface $u_z$")

def _draw_deformed_3d(ax, mesh, ux, uy, uz, uzn, L, W_, H, warp_frac=0.18):
    '''3D deformed plate: outer surface warped by the displacement, exaggerated.'''
    disp = np.vstack([ux, uy, uz]); umax = np.sqrt((disp**2).sum(0)).max() or 1.0
    Pdef = mesh.p + warp_frac*max(L, W_) * disp/umax
    fac = mesh.facets[:, mesh.boundary_facets()]          # outer skin (incl. notch walls)
    verts = np.transpose(Pdef[:, fac], (2, 1, 0))
    pc = Poly3DCollection(verts, cmap="RdBu_r", norm=plt.Normalize(-1.0, 1.0),
                          edgecolor="none")               # no per-facet edges -> clean & fast
    pc.set_array(uzn[fac].mean(0))
    ax.add_collection3d(pc)
    lo, hi = Pdef.min(1), Pdef.max(1)
    ax.set_xlim(lo[0], hi[0]); ax.set_ylim(lo[1], hi[1]); ax.set_zlim(lo[2], hi[2])
    ax.set_box_aspect((L, W_, 0.6*max(L, W_)))
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_title("3D deformed shape (exaggerated)")

def plot_mode(i, basis, mesh, vecs, freqs, L, W_, H, show3d=True):
    '''Show elastic mode i (1-based): 2D top-surface contour, plus a 3D deformed view if show3d.'''
    vec = vecs[:, i-1]
    ux = vec[basis.nodal_dofs[0]]
    uy = vec[basis.nodal_dofs[1]]
    uz = vec[basis.nodal_dofs[2]]
    uzn = uz / (np.max(np.abs(uz)) or 1.0)               # normalised u_z (symmetric -1..1)

    if show3d:
        fig = plt.figure(figsize=(11, 4.5), constrained_layout=True)
        ax1 = fig.add_subplot(1, 2, 1)
        ax2 = fig.add_subplot(1, 2, 2, projection="3d")
        _draw_contour(ax1, mesh, uzn, H)
        _draw_deformed_3d(ax2, mesh, ux, uy, uz, uzn, L, W_, H)
        cbar_axes = [ax1, ax2]
    else:
        fig = plt.figure(figsize=(6, 3.2), constrained_layout=True)
        ax1 = fig.add_subplot(1, 1, 1)
        _draw_contour(ax1, mesh, uzn, H)
        cbar_axes = [ax1]

    sm = plt.cm.ScalarMappable(norm=plt.Normalize(-1.0, 1.0), cmap="RdBu_r"); sm.set_array([])
    fig.suptitle(f"Mode {i}:  {freqs[i-1]:.1f} Hz")
    fig.colorbar(sm, ax=cbar_axes, shrink=0.6, ticks=np.linspace(-1, 1, 5),
                 label="normalised $u_z$")
    display(fig)        # display exactly once ...
    plt.close(fig)      # ... and close so the inline backend doesn't show it a second time

In [ ]:
def parse_notches(text):
    '''Parse the notch textarea: one "edge, pos, base, depth" (mm) per line.'''
    out = []
    for raw in text.splitlines():
        s = raw.strip()
        if not s or s.startswith("#"):
            continue
        parts = [p.strip() for p in s.split(",")]
        if len(parts) != 4:
            raise ValueError(f"bad notch line {raw!r} (need: edge, pos, base, depth)")
        edge = parts[0].lower()
        if edge not in ("bottom", "top", "left", "right"):
            raise ValueError(f"bad edge {edge!r} (use bottom/top/left/right)")
        try:
            pos, base, depth = (float(parts[1]), float(parts[2]), float(parts[3]))
        except ValueError:
            raise ValueError(f"notch line {raw!r}: pos/base/depth must be numbers (mm)")
        out.append(dict(edge=edge, pos=pos, base=base, depth=depth))
    return out

def validate_notches(notches_mm, L_mm, W_mm):
    '''Raise ValueError if any notch is off-edge, non-positive, too deep, or overlaps another.'''
    by_edge = {}
    for k, nc in enumerate(notches_mm, 1):
        edge, pos, base, depth = nc["edge"], nc["pos"], nc["base"], nc["depth"]
        if base <= 0 or depth <= 0:
            raise ValueError(f"notch {k} ({edge}): base and depth must be > 0.")
        along = L_mm if edge in ("bottom", "top") else W_mm   # pos runs along this edge
        into  = W_mm if edge in ("bottom", "top") else L_mm   # depth cuts into this span
        if pos - base/2 < 0 or pos + base/2 > along:
            raise ValueError(f"notch {k} ({edge}): pos={pos:g}, base={base:g} mm runs off the "
                             f"edge (must fit within 0..{along:g} mm).")
        if depth >= 0.5*into:
            raise ValueError(f"notch {k} ({edge}): depth={depth:g} mm >= half the {into:g} mm "
                             f"span — it would cut across the plate.")
        by_edge.setdefault(edge, []).append((pos, base, k))
    for edge, items in by_edge.items():
        items.sort()
        for (p1, b1, k1), (p2, b2, k2) in zip(items, items[1:]):
            if p2 - p1 < 0.5*(b1 + b2):
                raise ValueError(f"notches {k1} and {k2} on {edge} overlap "
                                 f"(centres {p1:g} and {p2:g} mm).")

def run_analysis(L_mm, W_mm, H_mm, material, customE, customRho, customNu,
                 f_max, n_modes, epw, enable_incuts, notches_mm,
                 do_convergence=False, do_compare=True, mode3d="selected", sel_mode=1):
    '''Full pipeline: size mesh -> mesh -> solve -> report table, plots, comparison, convergence.

    mode3d: "none" | "selected" | "all" controls which modes also get the (slower) 3D view.'''
    n_modes = max(1, int(n_modes))
    mat = dict(E=customE, rho=customRho, nu=customNu) if material == "custom" else MATERIALS[material]
    E, rho, nu = mat["E"], mat["rho"], mat["nu"]
    L, W_, H = L_mm/1000.0, W_mm/1000.0, H_mm/1000.0
    if enable_incuts:
        validate_notches(notches_mm, L_mm, W_mm)
    notches_m = ([dict(edge=n["edge"], pos=n["pos"]/1000.0, base=n["base"]/1000.0,
                       depth=n["depth"]/1000.0) for n in notches_mm] if enable_incuts else None)

    print(f"Plate : {L_mm:g} x {W_mm:g} x {H_mm:g} mm   |   material: {material}  "
          f"(E={E:.3e} Pa, rho={rho:g}, nu={nu:g})")
    print(f"Incuts: {'ON, ' + str(len(notches_m)) + ' notches' if notches_m else 'OFF (plain plate)'}\n")

    h_elem, _, lines = compute_element_size(L, W_, H, E, rho, nu, f_max, epw, notches_m)
    print("\n".join(lines))

    mesh = make_mesh(L, W_, H, h_elem, notches_m)
    print(f"\nMesh: {mesh.p.shape[1]} nodes, {mesh.t.shape[1]} tetrahedra\n")

    basis, freqs, vecs = solve_modes(mesh, E, rho, nu, n_modes, f_max)
    n_rigid, idx = split_rigid_elastic(freqs, n_modes)
    ef, ev = freqs[idx], vecs[:, idx]
    if n_rigid != 6:
        print(f"WARNING: expected 6 rigid-body modes, found {n_rigid} — check inputs.\n")

    print(f"Filtered {n_rigid} rigid-body modes. First {len(ef)} elastic modes:\n")
    print(f"{'mode':>4} | {'freq [Hz]':>11}")
    print("-"*20)
    for i, f in enumerate(ef, 1):
        print(f"{i:>4} | {f:>11.2f}{'   << above f_max' if f > f_max else ''}")
    if np.any(ef > f_max):
        print(f"\nNote: modes above f_max={f_max:g} Hz are under-resolved by this mesh "
              f"(raise f_max or density to trust them).")

    if enable_incuts and notches_m and do_compare:
        print("\n--- effect of incuts (plain vs notched) ---")
        pf = elastic_frequencies(make_mesh(L, W_, H, h_elem, None), E, rho, nu, n_modes, f_max)
        m = min(len(pf), len(ef))
        print(f"{'mode':>4} | {'plain [Hz]':>11} | {'notched [Hz]':>12} | {'delta':>8}")
        print("-"*45)
        for i in range(m):
            print(f"{i+1:>4} | {pf[i]:>11.2f} | {ef[i]:>12.2f} | {100*(ef[i]-pf[i])/pf[i]:>+7.2f}%")

    print(f"\nMode shapes (2D top-surface $u_z$; 3D deformed view: {mode3d}):")
    for i in range(1, len(ef) + 1):
        show3d = (mode3d == "all") or (mode3d == "selected" and i == int(sel_mode))
        plot_mode(i, basis, mesh, ev, ef, L, W_, H, show3d=show3d)

    if do_convergence:
        print("\n--- mesh-convergence check (re-solve at finer mesh) ---")
        mesh_f = make_mesh(L, W_, H, h_elem/1.5, notches_m)
        ff = elastic_frequencies(mesh_f, E, rho, nu, n_modes, f_max)[:5]
        print(f"coarse: {mesh.p.shape[1]} nodes   fine: {mesh_f.p.shape[1]} nodes\n")
        print(f"{'mode':>4} | {'coarse [Hz]':>12} | {'fine [Hz]':>11} | {'change':>8}")
        print("-"*46)
        for i in range(min(len(ef), len(ff))):
            print(f"{i+1:>4} | {ef[i]:>12.2f} | {ff[i]:>11.2f} | {100*(ff[i]-ef[i])/ef[i]:>7.2f}%")
        print("\n< ~1-2% change => converged.")
    print("\nDone.")

## Parameter guide

What each input means: the plate **Length × Width × Thickness**, and each incut line
`edge, pos, base, depth` (all in mm). `pos` is measured from the lower-left corner **along the
chosen edge**, `depth` points **into** the plate, and notches are cut through the **full
thickness**.

In [ ]:
# Diagram of what the input parameters mean (a schematic — not your actual dimensions).
from matplotlib.patches import Polygon as _Poly, Rectangle as _Rect

def draw_param_diagram():
    L, W_, H = 300.0, 120.0, 14.0          # example values, for the drawing only
    PLATE, CUT, DIM = "#dfe6ee", "#c0392b", "#333333"
    def _dim(ax, p1, p2, text, tpos=None, tcol=DIM, fs=10, rot=0):
        ax.annotate("", xy=p2, xytext=p1, arrowprops=dict(arrowstyle="<->", color=tcol, lw=1.3))
        if tpos is None: tpos = ((p1[0]+p2[0])/2, (p1[1]+p2[1])/2)
        ax.text(*tpos, text, color=tcol, fontsize=fs, ha="center", va="center", rotation=rot,
                bbox=dict(fc="white", ec="none", pad=0.5))
    def _notch(edge, pos, base, depth):
        if   edge == "bottom": O=np.array([pos,0]); n=np.array([0,1]); t=np.array([1,0])
        elif edge == "left":   O=np.array([0,pos]); n=np.array([1,0]); t=np.array([0,1])
        return np.array([O-0.5*base*t, O+0.5*base*t, O+depth*n])

    fig = plt.figure(figsize=(10.5, 6.2), constrained_layout=True)
    gs = fig.add_gridspec(2, 1, height_ratios=[3.2, 1])
    axT = fig.add_subplot(gs[0]); axS = fig.add_subplot(gs[1])

    # top view
    axT.add_patch(_Rect((0,0), L, W_, fc=PLATE, ec="black", lw=1.6))
    axT.text(L/2, W_/2, "TOP VIEW", ha="center", va="center", color="#5a6b7b",
             fontsize=13, fontweight="bold", alpha=.6)
    _dim(axT, (0,-30), (L,-30), "Length  (x)", tpos=(L/2,-44))
    _dim(axT, (-36,0), (-36,W_), "Width  (y)", tpos=(-54,W_/2), rot=90)
    axT.plot(0,0,"ko",ms=5); axT.text(-6,-10,"(0,0)",ha="right",va="top",fontsize=9,color=DIM)
    axT.text(L/2, W_+6, "edge = top", ha="center", va="bottom", fontsize=9, color="#777")
    axT.text(L/2, -6,  "edge = bottom", ha="center", va="top", fontsize=9, color="#777")
    axT.text(8, W_*0.82, "edge = left",  ha="left",  va="center", fontsize=9, color="#9aa7b3", rotation=90)
    axT.text(L-8, W_*0.5, "edge = right", ha="right", va="center", fontsize=9, color="#9aa7b3", rotation=90)
    p, b, d = 210.0, 44.0, 40.0
    axT.add_patch(_Poly(_notch("bottom", p, b, d), closed=True, fc="white", ec=CUT, lw=1.8))
    _dim(axT, (p-b/2, 8), (p+b/2, 8), "base", tpos=(p, 16), tcol=CUT)
    _dim(axT, (p, 0), (p, d), "depth", tpos=(p+20, d/2), tcol=CUT)
    _dim(axT, (0, -12), (p, -12), "pos", tpos=(p/2, -20))
    pL, bL, dL = 42.0, 34.0, 34.0
    axT.add_patch(_Poly(_notch("left", pL, bL, dL), closed=True, fc="white", ec=CUT, lw=1.8))
    _dim(axT, (-16, 0), (-16, pL), "pos", tpos=(-16, pL/2), rot=90)
    _dim(axT, (0, pL), (dL, pL), "depth", tpos=(dL/2, pL+12), tcol=CUT, fs=9)
    axT.set_xlim(-70, L+40); axT.set_ylim(-55, W_+22); axT.set_aspect("equal"); axT.axis("off")

    # side view (thickness)
    axS.add_patch(_Rect((0,0), L, H, fc=PLATE, ec="black", lw=1.6))
    axS.add_patch(_Rect((p-b/2, 0), b, H, fc="white", ec=CUT, lw=1.4, hatch="//"))
    _dim(axS, (-22, 0), (-22, H), "Thickness", tpos=(-46, H/2), rot=90)
    axS.text(L/2, H+5, "SIDE VIEW  —  notches are cut through the full thickness",
             ha="center", va="bottom", fontsize=10, color="#5a6b7b")
    axS.set_xlim(-70, L+40); axS.set_ylim(-6, H+22); axS.set_aspect("auto"); axS.axis("off")

    fig.suptitle("What the inputs mean — Length × Width × Thickness, and each notch = edge, pos, base, depth (mm)",
                 fontsize=11)
    display(fig); plt.close(fig)

draw_param_diagram()

## Control panel

Set the parameters and press **▶ Run**. Incuts are one per line — `edge, pos, base, depth`
(mm), where `edge` is `bottom`/`top`/`left`/`right`. **3D view** chooses whether the (slightly
slower) 3D deformed plot is drawn for *none* / the *selected* mode / *all* modes. The
plain-vs-notched comparison and the mesh-convergence check each add one extra solve.

In [ ]:
# Close widgets created by a previous run of THIS cell, so their comms are torn down and a
# single click can't be delivered to stale buttons (avoids a click running 2-3x after several
# "Run All"s without a kernel restart).
for _n in ('panel', 'out', 'run_btn', 'status', 'w_L', 'w_W', 'w_H', 'w_mat', 'w_E', 'w_rho',
           'w_nu', 'w_fmax', 'w_nm', 'w_epw', 'w_incut', 'w_cmp', 'w_conv', 'w_notch',
           'w_mode3d', 'w_sel'):
    _old = globals().get(_n)
    if _old is not None:
        try:
            _old.close()
        except Exception:
            pass

_S  = {'description_width': '92px'}
_LW = W.Layout(width='205px')

def _hdr(text):
    return W.HTML(f"<b style='font-size:108%'>{text}</b>")
def _cap(text):
    return W.HTML(f"<span style='color:#666;font-size:90%'>{text}</span>")

# --- plate geometry ---
w_L = W.BoundedFloatText(value=300.0, min=1, max=5000, step=10, description='Length [mm]', style=_S, layout=_LW)
w_W = W.BoundedFloatText(value=100.0, min=1, max=5000, step=10, description='Width [mm]',  style=_S, layout=_LW)
w_H = W.BoundedFloatText(value=10.0,  min=0.5, max=1000, step=1, description='Thick. [mm]', style=_S, layout=_LW)

# --- material (E/rho/nu editable only for 'custom') ---
w_mat = W.Dropdown(options=['steel', 'bronze', 'aluminum', 'custom'], value='steel',
                   description='Material', style=_S, layout=_LW)
w_E   = W.BoundedFloatText(value=200.0, min=1, max=1000,  step=1,  description='E [GPa]',      style=_S, layout=_LW)
w_rho = W.BoundedFloatText(value=7800., min=100, max=25000, step=50, description='rho [kg/m³]', style=_S, layout=_LW)
w_nu  = W.BoundedFloatText(value=0.30,  min=0.0, max=0.49, step=0.01, description='nu',         style=_S, layout=_LW)

# --- solver ---
w_fmax = W.BoundedFloatText(value=2000.0, min=100, max=50000, step=100, description='f_max [Hz]', style=_S, layout=_LW)
w_nm   = W.BoundedIntText(value=6, min=1, max=12, description='# modes', style=_S, layout=_LW)
w_epw  = W.BoundedIntText(value=8, min=4, max=16, description='el/wave', style=_S, layout=_LW)

# --- options & 3D display ---
w_incut = W.Checkbox(value=True,  description='Enable incuts', indent=False)
w_cmp   = W.Checkbox(value=True,  description='Compare plain vs notched', indent=False)
w_conv  = W.Checkbox(value=False, description='Mesh-convergence check (slower)', indent=False)
w_mode3d = W.Dropdown(options=['none', 'selected', 'all'], value='selected',
                      description='3D view', style=_S, layout=_LW)
w_sel    = W.BoundedIntText(value=1, min=1, max=12, description='3D mode #', style=_S, layout=_LW)

w_notch = W.Textarea(
    value="bottom, 75, 25, 15\nbottom, 225, 25, 15\ntop, 75, 25, 15\ntop, 225, 25, 15\n"
          "left, 25, 20, 15\nleft, 75, 20, 15\nright, 25, 20, 15\nright, 75, 20, 15",
    layout=W.Layout(width='330px', height='150px'))

run_btn = W.Button(description='▶ Run', button_style='success',
                   layout=W.Layout(width='150px', height='40px'))
status  = W.HTML("")
out     = W.Output()

# enable E/rho/nu only for custom; otherwise show the preset values (read-only)
def _sync_material(*_):
    custom = (w_mat.value == 'custom')
    for wdg in (w_E, w_rho, w_nu):
        wdg.disabled = not custom
    if not custom:
        p = MATERIALS[w_mat.value]
        w_E.value, w_rho.value, w_nu.value = p['E']/1e9, p['rho'], p['nu']
w_mat.observe(_sync_material, names='value')
_sync_material()

_running = {'busy': False}

def _on_run(_):
    if _running['busy']:                      # ignore duplicate/re-entrant clicks
        return
    _running['busy'] = True
    run_btn.disabled = True; run_btn.description = '⏳ Running…'; run_btn.button_style = 'warning'
    status.value = "<span style='color:#b58900'>Running… meshing + solving</span>"
    try:
        with out:
            out.clear_output(wait=True)
            try:
                notches = parse_notches(w_notch.value) if w_incut.value else []
                run_analysis(w_L.value, w_W.value, w_H.value, w_mat.value,
                             w_E.value*1e9, w_rho.value, w_nu.value,   # E entered in GPa -> Pa
                             w_fmax.value, w_nm.value, w_epw.value,
                             w_incut.value, notches,
                             do_convergence=w_conv.value, do_compare=w_cmp.value,
                             mode3d=w_mode3d.value, sel_mode=w_sel.value)
                status.value = "<span style='color:#2aa198'>✓ Done</span>"
            except (ValueError, RuntimeError) as e:           # expected / friendly errors
                print(f"⚠  {e}")
                status.value = "<span style='color:#dc322f'>⚠ Input/geometry problem — see message</span>"
            except Exception:                                 # unexpected -> full traceback
                traceback.print_exc()
                status.value = "<span style='color:#dc322f'>✗ Unexpected error — see traceback</span>"
    finally:
        run_btn.disabled = False; run_btn.description = '▶ Run'; run_btn.button_style = 'success'
        _running['busy'] = False

run_btn.on_click(_on_run)

_left = W.VBox([
    _hdr('Plate geometry'),          W.HBox([w_L, w_W, w_H]),
    _hdr('Material'),                W.HBox([w_mat, w_E, w_rho, w_nu]),
    _cap('E / rho / nu are editable only when Material = “custom”.'),
    _hdr('Solver'),                  W.HBox([w_fmax, w_nm, w_epw]),
    _hdr('Display'),                 W.HBox([w_mode3d, w_sel]),
    _cap('3D view: none = 2D only (fastest) · selected = 3D for the chosen mode · all = every mode.'),
])
_right = W.VBox([
    _hdr('Incuts'),
    _cap('One per line:  edge, pos, base, depth  (mm).<br>edge ∈ bottom / top / left / right.'),
    w_notch,
    W.VBox([w_incut, w_cmp, w_conv]),
])

panel = W.VBox([
    W.HTML("<h3 style='margin:2px'>Bell-plate modal analysis</h3>"
           "<span style='color:#666'>Set parameters, then press ▶ Run.</span>"),
    W.HBox([_left, _right]),
    W.HBox([run_btn, status]),
])
display(panel, out)